In [0]:
# =====================================================================
# proceso / 04_transform.py
# Task "transform" (Silver). PySpark puro (sin Spark SQL).
# Depende de las 3 ingestas bronze. A diferencia del diseño inicial, NO
# se unen Superstore y E-commerce en una sola tabla: tienen grano
# distinto (Superstore = 1 fila por linea de orden; E-commerce = 1 fila
# por dia, sin Order_ID/Customer_ID/Region reales). Forzar esa union
# implicaria inventar IDs y dejar dimensiones vacias en fact_sales, asi
# que cada fuente se limpia en su propia tabla silver, y la comparacion
# entre canales se resuelve en golden, donde ambas SI comparten un
# grano comun (mes + categoria, y feriado/no feriado).
# =====================================================================

In [0]:
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("catalogo", "retail_medallion")
catalogo = dbutils.widgets.get("catalogo")

In [0]:
calendar_clean = (
    spark.table(f"{catalogo}.bronze.calendar_holidays")
    .dropDuplicates(["Date"])
    .withColumn("holiday_date", F.expr("try_to_date(Date, 'yyyy-MM-dd')"))
    .select(F.col("holiday_date"), F.col("Holiday").alias("holiday_name"))
)

# --- Superstore (grano: linea de orden) ---------------------------------
bronze_superstore = spark.table(f"{catalogo}.bronze.superstore_raw")

In [0]:
superstore_clean = (
    bronze_superstore
    .dropDuplicates(["Row_ID"])  # Row_ID es la llave real; (Order_ID,Product_ID) descartaria 8 lineas legitimas verificadas en el CSV real
    .filter(F.col("Sales").isNotNull() & (F.col("Sales") > 0))
    .withColumn("order_date", F.expr("try_to_date(Order_Date, 'M/d/yyyy')"))  # try_to_date: fecha corrupta -> NULL en vez de tumbar el job (modo ANSI)
    .withColumn("ship_date", F.expr("try_to_date(Ship_Date, 'M/d/yyyy')"))
    .filter(F.col("order_date").isNotNull())  # descarta filas cuya fecha no se pudo parsear (dato corrupto en la fuente)
    .withColumn("lead_time_days", F.datediff(F.col("ship_date"), F.col("order_date")))
    .withColumn("profit_margin_pct", F.round(F.col("Profit") / F.col("Sales"), 4))
    .join(calendar_clean, F.col("order_date") == calendar_clean.holiday_date, "left")
    .withColumn("is_holiday", F.col("holiday_date").isNotNull())
    .withColumn("_transform_timestamp", F.current_timestamp())
    .select(
        F.col("Order_ID").alias("order_id"),
        F.col("order_date"),
        F.col("Customer_ID").alias("customer_id"),
        F.col("Segment").alias("customer_segment"),
        F.col("Region").alias("region"),
        F.col("Product_ID").alias("product_id"),
        F.col("Category").alias("category"),
        F.col("Sub_Category").alias("sub_category"),
        F.col("Sales").alias("sales_amount"),
        F.col("Quantity").alias("quantity"),
        F.col("Discount").alias("discount_pct"),
        F.col("Profit").alias("profit_amount"),
        F.col("lead_time_days"),
        F.col("profit_margin_pct"),
        F.col("is_holiday"),
        F.col("holiday_name"),
        F.col("_transform_timestamp"),
    )
)

superstore_clean.write.mode("overwrite").insertInto(f"{catalogo}.silver.superstore_clean")
print(f"Silver OK -> {catalogo}.silver.superstore_clean ({superstore_clean.count()} filas)")

Silver OK -> retail_medallion.silver.superstore_clean (9994 filas)


In [0]:
# --- E-commerce (grano: dia + categoria, SIN Order_ID/Customer_ID/Region) -
# Discount viene en escala 0-50 (porcentaje directo, no fraccion) -> /100.
# sales_amount se deriva (Price x Units_Sold), no viene explicito.
# profit_amount se estima como ingreso neto de descuento menos el gasto
# de marketing del dia: es una aproximacion (ignora costo de producto),
# pero a diferencia de asumir un margen fijo inventado, usa datos reales
# de la fuente (Marketing_Spend) en vez de un porcentaje arbitrario.
bronze_ecommerce = spark.table(f"{catalogo}.bronze.ecommerce_raw")

ecommerce_clean = (
    bronze_ecommerce
    .dropDuplicates(["Date"])
    .filter(F.col("Price").isNotNull() & (F.col("Price") > 0) & F.col("Units_Sold").isNotNull())
    .withColumn("order_date", F.expr("try_to_date(Date, 'd-M-yyyy')"))  # try_to_date: consistente con superstore
    .filter(F.col("order_date").isNotNull())
    .withColumn("sales_amount", F.round(F.col("Price") * F.col("Units_Sold"), 2))
    .withColumn("discount_pct", F.round(F.col("Discount") / 100.0, 4))
    .withColumn(
        "profit_amount",
        F.round(F.col("sales_amount") * (F.lit(1) - F.col("discount_pct")) - F.col("Marketing_Spend"), 2),
    )
    .withColumn("profit_margin_pct", F.round(F.col("profit_amount") / F.col("sales_amount"), 4))
    .join(calendar_clean, F.col("order_date") == calendar_clean.holiday_date, "left")
    .withColumn("is_holiday", F.col("holiday_date").isNotNull())
    .withColumn("_transform_timestamp", F.current_timestamp())
    .select(
        F.col("order_date"),
        F.col("Product_Category").alias("category"),
        F.col("Customer_Segment").alias("customer_segment"),
        F.col("sales_amount"),
        F.col("Units_Sold").alias("units_sold"),
        F.col("discount_pct"),
        F.col("Marketing_Spend").alias("marketing_spend"),
        F.col("profit_amount"),
        F.col("profit_margin_pct"),
        F.col("is_holiday"),
        F.col("holiday_name"),
        F.col("_transform_timestamp"),
    )
)

ecommerce_clean.write.mode("overwrite").insertInto(f"{catalogo}.silver.ecommerce_daily_clean")
print(f"Silver OK -> {catalogo}.silver.ecommerce_daily_clean ({ecommerce_clean.count()} filas)")

Silver OK -> retail_medallion.silver.ecommerce_daily_clean (0 filas)
